# TerraMind on SmallMinesDS — binary mine segmentation

**Question (RQ1).** Does adding Sentinel-1 SAR to Sentinel-2 optical imagery improve
binary segmentation of artisanal gold mining in south-western Ghana?

**This notebook runs two arms.**

| Arm | Input | Bands |
|-----|-------|-------|
| `A1` | Sentinel-2 L2A | 10 |
| `A4` | Sentinel-2 + Sentinel-1 RTC | 12 |

Same model, same split, same schedule — one difference: two radar channels.

**Setup.**
- Target: binary, `0` = no mining, `1` = mining. Nodata → `ignore_index`.
- Sample: one *(tile, year)* patch. 2016 and 2022 are pooled as independent samples;
  the train/val/test split is assigned **by tile** so the same ground never straddles
  train and test.
- Model: `terramind_v1_base` encoder, **frozen** → UNet decoder trained from scratch.
- Normalisation: TerraMind's pretraining statistics by default. Set
  `NORMALISATION = "dataset"` to re-run with statistics computed from the training
  split — this is the check for the radar offset described at the bottom.

Dataset: Ofori-Ampofo et al., *SmallMinesDS*, IEEE GRSL 2025 (`ellaampy/SmallMinesDS`).
Model: *TerraMind* v1 (IBM/ESA), `ibm-esa-geospatial/TerraMind-1.0-base`.

In [ ]:
!pip install -q huggingface_hub rasterio "terratorch>=1.2.13" "setuptools<81" torchgeo==0.9.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

In [ ]:
import os

PROJECT_ROOT = "drive/MyDrive/HonoursProject/IllegalMiningDetection"

os.chdir(PROJECT_ROOT)

## 0. Configuration

`SMOKE_TEST = True` runs everything end-to-end on 16 tiles with the tiny model, to prove the
pipeline before spending GPU time.

In [ ]:
SMOKE_TEST = False          # True: tiny model, 16 samples, 1 epoch
BACKBONE = "terramind_v1_tiny" if SMOKE_TEST else "terramind_v1_base"
EPOCHS = 1 if SMOKE_TEST else 30
BATCH_SIZE = 4 if SMOKE_TEST else 8   # halved token count vs the bi-temporal run; raise to 16 if VRAM allows
SEED = 42

# "terramind" -> the encoder's own pretraining statistics (correct for a FROZEN encoder)
# "dataset"   -> statistics from the training split (the radar-offset check)
NORMALISATION = "terramind"

In [ ]:
import importlib, shutil, sys, time, warnings, zipfile
from pathlib import Path

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

import terratorch  # noqa: F401  (registers backbones)
from terratorch.tasks import SemanticSegmentationTask

warnings.filterwarnings("ignore", category=UserWarning)
L.seed_everything(SEED, workers=True)
plt.rcParams["figure.dpi"] = 110

IN_COLAB = "google.colab" in sys.modules
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from importlib.metadata import version
print(f"torch {torch.__version__} | terratorch {version('terratorch')} | device {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

## 1. Data

The dataset is copied onto Colab's **local disk** before training. Reading ~4,300 small
GeoTIFFs per epoch across the Drive FUSE mount is the single biggest source of slowness —
it is what made the previous run crawl. The copy is a one-off couple of minutes and makes
every epoch afterwards disk-bound rather than network-bound.

In [ ]:
def acquire_dataset(root=Path(".")):
    for cand in (root / "Dataset", root / "SmallMinesDS" / "SmallMinesDS"):
        if (cand / "2016" / "IMAGE").is_dir():
            return cand
    from huggingface_hub import snapshot_download
    hf_dir = Path(snapshot_download(repo_id="ellaampy/SmallMinesDS", repo_type="dataset",
                                    local_dir=root / "SmallMinesDS"))
    with zipfile.ZipFile(hf_dir / "SmallMinesDS.zip") as z:
        z.extractall(hf_dir)
    return hf_dir / "SmallMinesDS"


def localise(src, dst=Path("/content/smallmines_local")):
    """One-off copy of the image tree to local disk. No-op off Colab or already local."""
    src = Path(src)
    if not IN_COLAB or "/drive/" not in str(src.resolve()):
        return src
    if not dst.exists():
        print("copying dataset to local disk (one-off, ~2 min) ...")
        shutil.copytree(src, dst)
        print("done")
    return dst


DRIVE_ROOT = acquire_dataset()
DATA_ROOT = localise(DRIVE_ROOT)
SPLITS_DIR = next(p for p in (Path("data_splits"), Path("SmallMinesDS/data_splits"))
                  if (p / "train_test_splits_2016.csv").is_file())
print("images:", DATA_ROOT)
print("splits:", SPLITS_DIR.resolve())

### 1.1 Data pipeline module

Written to `smallmines_data.py` so it can be imported by scripts and configs as well as the
notebook. Edit the cell, not the file.

In [ ]:
%%writefile smallmines_data.py
"""SmallMinesDS -> binary, single-epoch segmentation pipeline.

Each sample is ONE (tile, year) patch: 13 bands x 128 x 128 with a binary mine mask.
This module

- maps the 13 bands onto TerraMind's modalities (S2L2A, S1RTC, DEM),
- standardises with either TerraMind's pretraining statistics (for the frozen
  foundation-model encoder) or statistics computed from the training split
  (for models trained from scratch) - normalisation belongs to the MODEL,
- pools 2016 and 2022 as independent samples but assigns the train/val/test
  split BY TILE, so the same ground never straddles train and test.

Target is binary: 0 = no mining, 1 = mining, nodata -> IGNORE_INDEX.

Written out from terramind.ipynb via %%writefile; edit the notebook cell.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import torch
from lightning import LightningDataModule
from torch.utils.data import DataLoader, Dataset

YEARS = ("2016", "2022")
IMAGE_NODATA = -9999.0
MASK_NODATA = 255
IGNORE_INDEX = -1

CLASS_NAMES = ["no mining", "mining"]
NUM_CLASSES = 2

# Band layout of the 13-band SmallMinesDS GeoTIFF in TerraMind's band vocabulary.
MODALITY_BANDS = {
    "S2L2A": ["BLUE", "GREEN", "RED", "RED_EDGE_1", "RED_EDGE_2", "RED_EDGE_3",
              "NIR_BROAD", "NIR_NARROW", "SWIR_1", "SWIR_2"],
    "S1RTC": ["VV", "VH"],
    "DEM": ["DEM"],
}
MODALITY_INDICES = {"S2L2A": list(range(0, 10)), "S1RTC": [10, 11], "DEM": [12]}
ALL_MODALITIES = tuple(MODALITY_BANDS)

# TerraMind v1 pretraining statistics. S2L2A is the 12-band list; the two bands this
# dataset lacks (COASTAL_AEROSOL, WATER_VAPOR) are dropped by _subset.
_TM_S2L2A_BANDS = ["COASTAL_AEROSOL", "BLUE", "GREEN", "RED", "RED_EDGE_1", "RED_EDGE_2",
                   "RED_EDGE_3", "NIR_BROAD", "NIR_NARROW", "WATER_VAPOR", "SWIR_1", "SWIR_2"]
_TM_MEAN = {
    "S2L2A": [1390.458, 1503.317, 1718.197, 1853.910, 2199.100, 2779.975,
              2987.011, 3083.234, 3132.220, 3162.988, 2424.884, 1857.648],
    "S1RTC": [-10.93, -17.329],
    "DEM": [670.665],
}
_TM_STD = {
    "S2L2A": [2106.761, 2141.107, 2038.973, 2134.138, 2085.321, 1889.926,
              1820.257, 1871.918, 1753.829, 1797.379, 1434.261, 1334.311],
    "S1RTC": [4.391, 4.459],
    "DEM": [951.272],
}


def _subset(stats, modality):
    if modality == "S2L2A":
        return [stats[_TM_S2L2A_BANDS.index(b)] for b in MODALITY_BANDS["S2L2A"]]
    return stats


TERRAMIND_MEAN = {m: np.array(_subset(_TM_MEAN[m], m), dtype=np.float32) for m in ALL_MODALITIES}
TERRAMIND_STD = {m: np.array(_subset(_TM_STD[m], m), dtype=np.float32) for m in ALL_MODALITIES}


# --------------------------------------------------------------------------- paths

def find_data_root(start=Path(".")):
    start = Path(start)
    for cand in (start / "Dataset", start / "SmallMinesDS" / "SmallMinesDS", start / "SmallMinesDS"):
        if all((cand / y / "IMAGE").is_dir() and (cand / y / "MASK").is_dir() for y in YEARS):
            return cand
    raise FileNotFoundError(f"no <year>/IMAGE folders found under {start.resolve()}")


def find_splits_dir(start=Path(".")):
    start = Path(start)
    for cand in (start / "data_splits", start / "SmallMinesDS" / "data_splits"):
        if (cand / "train_test_splits_2016.csv").is_file():
            return cand
    raise FileNotFoundError(f"no data_splits/ folder found under {start.resolve()}")


def patch_paths(data_root, year, tile):
    data_root = Path(data_root)
    return (data_root / year / "IMAGE" / f"IMG_GH_{tile}_{year}.tif",
            data_root / year / "MASK" / f"MASK_GH_{tile}_{year}.tif")


# --------------------------------------------------------------------------- split

def make_pooled_split(splits_dir, out_csv, seed=42, val_frac=0.15, test_frac=0.15):
    """One row per (tile, year); split assigned BY TILE.

    2016 and 2022 are independent training samples, but the same 128x128 patch six
    years apart shares terrain, roads, field boundaries and most land cover. A
    sample-level split would put one year in train and the other in test for the same
    ground, so the test score would be partly a memorisation score. Splitting by tile
    keeps both years of a location together.

    Stratified on each tile's mean mine coverage across the two years, so rare
    high-coverage tiles are spread evenly.
    """
    from sklearn.model_selection import train_test_split

    rows = []
    for year in YEARS:
        df = pd.read_csv(Path(splits_dir) / f"train_test_splits_{year}.csv")
        df["tile"] = df["patch_name"].str.extract(r"_(\d{4})_")
        rows.append(df.assign(year=year)[["tile", "year", "class_percentage"]])
    samples = pd.concat(rows, ignore_index=True).dropna(subset=["tile"])

    tiles = samples.groupby("tile")["class_percentage"].mean().rename("pct").reset_index()
    tiles["bin"] = pd.qcut(tiles["pct"].rank(method="first"), 5, labels=False)

    trval, test = train_test_split(tiles, test_size=test_frac, random_state=seed,
                                   stratify=tiles["bin"])
    train, val = train_test_split(trval, test_size=val_frac / (1 - test_frac),
                                  random_state=seed, stratify=trval["bin"])
    assign = {t: "train" for t in train.tile}
    assign.update({t: "val" for t in val.tile})
    assign.update({t: "test" for t in test.tile})

    samples["split"] = samples.tile.map(assign)
    samples = samples.sort_values(["tile", "year"]).reset_index(drop=True)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    samples.to_csv(out_csv, index=False)
    return samples


# --------------------------------------------------------------------------- statistics

def compute_dataset_stats(data_root, samples, modalities=ALL_MODALITIES,
                          max_samples=300, seed=0):
    """Per-band mean/std over a subsample of the TRAINING split, ignoring nodata.

    Used for models trained from scratch, and as the alternative normalisation for
    the frozen encoder when checking whether the radar offset matters.
    """
    rng = np.random.default_rng(seed)
    samples = [tuple(s) for s in samples]
    if len(samples) > max_samples:
        pick = rng.choice(len(samples), max_samples, replace=False)
        samples = [samples[i] for i in pick]

    sums, sqs, counts = {}, {}, {}
    for tile, year in samples:
        img_path, _ = patch_paths(data_root, year, tile)
        with rasterio.open(img_path) as src:
            img = src.read().astype(np.float64)
        for mod in modalities:
            x = img[MODALITY_INDICES[mod]]
            ok = x != IMAGE_NODATA
            sums.setdefault(mod, np.zeros(x.shape[0]))
            sqs.setdefault(mod, np.zeros(x.shape[0]))
            counts.setdefault(mod, np.zeros(x.shape[0]))
            sums[mod] += np.where(ok, x, 0.0).sum(axis=(1, 2))
            sqs[mod] += np.where(ok, x ** 2, 0.0).sum(axis=(1, 2))
            counts[mod] += ok.sum(axis=(1, 2))

    mean, std = {}, {}
    for mod in modalities:
        n = np.maximum(counts[mod], 1)
        m = sums[mod] / n
        v = np.maximum(sqs[mod] / n - m ** 2, 1e-8)
        mean[mod] = m.astype(np.float32)
        std[mod] = np.sqrt(v).astype(np.float32)
    return mean, std


# --------------------------------------------------------------------------- dataset

class SmallMinesDataset(Dataset):
    """One item = one (tile, year) patch.

    Returns {"image": {modality: (C, H, W) float tensor}, "mask": (H, W) long tensor,
    "filename": "<tile>_<year>"}.
    """

    def __init__(self, data_root, samples, modalities=ALL_MODALITIES,
                 mean=None, std=None, augment=False):
        self.data_root = Path(data_root)
        self.samples = [tuple(s) for s in samples]
        self.modalities = list(modalities)
        self.mean = mean if mean is not None else TERRAMIND_MEAN
        self.std = std if std is not None else TERRAMIND_STD
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def _read(self, year, tile):
        img_path, mask_path = patch_paths(self.data_root, year, tile)
        with rasterio.open(img_path) as src:
            image = src.read().astype(np.float32)
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
        return image, mask

    def __getitem__(self, i):
        tile, year = self.samples[i]
        img, mask = self._read(year, tile)

        target = (mask == 1).astype(np.int64)
        target[mask == MASK_NODATA] = IGNORE_INDEX

        sample = {}
        for mod in self.modalities:
            x = img[MODALITY_INDICES[mod]].copy()           # (C, H, W)
            nodata = x == IMAGE_NODATA
            x = (x - self.mean[mod][:, None, None]) / self.std[mod][:, None, None]
            x[nodata] = 0.0                                  # nodata -> the mean
            sample[mod] = x

        if self.augment:
            sample, target = self._d4(sample, target)

        return {
            "image": {m: torch.from_numpy(np.ascontiguousarray(x)) for m, x in sample.items()},
            "mask": torch.from_numpy(np.ascontiguousarray(target)),
            "filename": f"{tile}_{year}",
        }

    @staticmethod
    def _d4(sample, target):
        """Random dihedral flip/rotation, applied identically to every modality."""
        k = np.random.randint(4)
        flip = np.random.rand() < 0.5
        for mod, x in sample.items():
            x = np.rot90(x, k, axes=(-2, -1))
            sample[mod] = x[..., ::-1] if flip else x
        target = np.rot90(target, k, axes=(-2, -1))
        return sample, (target[..., ::-1] if flip else target)


class SmallMinesDataModule(LightningDataModule):
    def __init__(self, data_root, split_csv, modalities=ALL_MODALITIES, mean=None, std=None,
                 batch_size=8, num_workers=2, augment=True, limit=None):
        super().__init__()
        self.data_root = Path(data_root)
        self.split_csv = Path(split_csv)
        self.modalities = list(modalities)
        self.mean = mean
        self.std = std
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.augment = augment
        self.limit = limit
        self.samples = {}

    def setup(self, stage=None):
        df = pd.read_csv(self.split_csv, dtype={"tile": str, "year": str})
        for split in ("train", "val", "test"):
            rows = df.loc[df.split == split, ["tile", "year"]].itertuples(index=False, name=None)
            rows = list(rows)
            self.samples[split] = rows[: self.limit] if self.limit else rows

    def _loader(self, split, shuffle):
        ds = SmallMinesDataset(self.data_root, self.samples[split], self.modalities,
                               mean=self.mean, std=self.std,
                               augment=self.augment and split == "train")
        return DataLoader(ds, batch_size=self.batch_size, shuffle=shuffle,
                          num_workers=self.num_workers, pin_memory=True,
                          persistent_workers=self.num_workers > 0, drop_last=shuffle)

    def train_dataloader(self):
        return self._loader("train", shuffle=True)

    def val_dataloader(self):
        return self._loader("val", shuffle=False)

    def test_dataloader(self):
        return self._loader("test", shuffle=False)

    def predict_dataloader(self):
        return self.test_dataloader()


# --------------------------------------------------------------------------- diagnostics

def haze_proxy(image, mask):
    """Mean blue reflectance over valid NON-mine pixels: a thin-cloud / haze stand-in.

    Computed off-mine because bright alluvial mining ponds mimic cloud in the visible
    bands (Gallwey et al. 2020) - a whole-tile brightness measure would partly be
    measuring the target class, which would make any stratification circular.
    """
    blue = image[0]
    ok = (blue != IMAGE_NODATA) & (mask != MASK_NODATA) & (mask != 1)
    return float(blue[ok].mean()) if ok.any() else np.nan

In [ ]:
import smallmines_data
importlib.reload(smallmines_data)
from smallmines_data import (ALL_MODALITIES, CLASS_NAMES, IGNORE_INDEX, MODALITY_BANDS,
                             NUM_CLASSES, TERRAMIND_MEAN, TERRAMIND_STD,
                             SmallMinesDataModule, SmallMinesDataset,
                             compute_dataset_stats, haze_proxy, make_pooled_split)

### 1.2 One split per tile

2016 and 2022 are pooled as independent training samples, but the split is assigned **by
tile**. The same 128 × 128 patch six years apart shares terrain, roads, field boundaries and
most of its land cover — a sample-level split would put one year in train and the other in
test for the same ground, and the test score would become partly a memorisation score.

Separately: the published `train_test_splits_<year>.csv` files were stratified independently
per year and **disagree for 42 % of tiles**, so pooling on them would leak in exactly this
way. The split below is drawn fresh, seeded, and stratified on each tile's mean mine
coverage.

In [ ]:
SPLIT_CSV = SPLITS_DIR / "pooled_splits.csv"
samples = make_pooled_split(SPLITS_DIR, SPLIT_CSV, seed=SEED)

print(f"{samples.tile.nunique()} tiles -> {len(samples)} samples  ->  {SPLIT_CSV}")
display(pd.crosstab(samples["split"], samples["year"], margins=True))
print("\nmean mine coverage by split (%):")
display(samples.groupby("split")["class_percentage"].agg(["mean", "median", "max"]).round(2))

### 1.3 Normalisation and a look at the bands

Normalisation is a property of the **model**, not the dataset: a frozen encoder must see the
distribution its weights were fitted to, so it gets TerraMind's pretraining statistics.
Models trained from scratch would get dataset statistics instead.

The printout below is diagnostic. After standardisation each band should sit near mean 0,
std 1. Watch two rows in particular — they are flagged automatically.

In [ ]:
if NORMALISATION == "dataset":
    tmp = SmallMinesDataModule(DATA_ROOT, SPLIT_CSV)
    tmp.setup()
    MEAN, STD = compute_dataset_stats(DATA_ROOT, tmp.samples["train"])
    print("using DATASET statistics\n")
else:
    MEAN, STD = TERRAMIND_MEAN, TERRAMIND_STD
    print("using TERRAMIND pretraining statistics\n")

datamodule = SmallMinesDataModule(
    DATA_ROOT, SPLIT_CSV, modalities=ALL_MODALITIES, mean=MEAN, std=STD,
    batch_size=BATCH_SIZE, num_workers=0 if SMOKE_TEST else 2,
    limit=16 if SMOKE_TEST else None,
)
datamodule.setup()
print({k: len(v) for k, v in datamodule.samples.items()})

batch = next(iter(datamodule.val_dataloader()))
print()
for mod, x in batch["image"].items():
    m, s = float(x.mean()), float(x.std())
    flag = ""
    if abs(m) > 1.0:
        flag = "  <-- far from 0: this modality is out of distribution for the encoder"
    elif s < 0.1:
        flag = "  <-- near-constant: this band carries almost no signal after standardising"
    print(f"{mod:6s} {tuple(x.shape)}  mean {m:+.2f}  std {s:.2f}{flag}")

print("\nmask  ", tuple(batch["mask"].shape), batch["mask"].dtype)
valid = batch["mask"][batch["mask"] >= 0]
counts = torch.bincount(valid.flatten(), minlength=NUM_CLASSES)
print("class pixel share in this batch:",
      {n: f"{c / counts.sum():.2%}" for n, c in zip(CLASS_NAMES, counts)})

Sanity check: true-colour composite and binary target for the most heavily mined validation
sample.

In [ ]:
from matplotlib.colors import ListedColormap
CLASS_CMAP = ListedColormap(["#f2f2f2", "#b5342a"])


def stretch(band, low=2, high=98):
    band = np.asarray(band, dtype=np.float32)
    ok = band[np.isfinite(band) & (band != -9999)]
    if ok.size == 0:
        return np.zeros_like(band)
    lo, hi = np.percentile(ok, [low, high])
    return np.clip((band - lo) / max(hi - lo, 1e-6), 0, 1)


def rgb(image):
    return np.dstack([stretch(image[b]) for b in (2, 1, 0)])


def show_sample(tile, year, pred=None):
    ds = SmallMinesDataset(DATA_ROOT, [(tile, year)])
    img, mask = ds._read(year, tile)
    target = (mask == 1).astype(int)
    target[mask == 255] = -1
    panels = [(f"{tile} / {year}", rgb(img)), ("target", target)]
    if pred is not None:
        panels.append(("prediction", pred))
    fig, axes = plt.subplots(1, len(panels), figsize=(3.5 * len(panels), 3.7))
    for ax, (title, arr) in zip(np.atleast_1d(axes), panels):
        if arr.ndim == 2:
            ax.imshow(np.ma.masked_where(arr < 0, arr), cmap=CLASS_CMAP, vmin=0, vmax=1,
                      interpolation="nearest")
        else:
            ax.imshow(arr)
        ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("grey = no mining | red = mining", fontsize=9)
    plt.tight_layout()


top = samples.loc[samples.split == "val"].sort_values("class_percentage").iloc[-1]
show_sample(top.tile, top.year)

## 2. Model

`EncoderDecoderFactory` assembles: TerraMind encoder (pretrained, band subset selected by name
so the 10 available S2 bands pick up their pretrained patch-embedding weights) → necks that
reshape four transformer layers into a feature pyramid → `UNetDecoder` → 2-class head.

`freeze_backbone=True` means **only the decoder and head train** — what is being measured is
TerraMind's pretrained representation, not how good a TerraMind-based model could get.

Dice loss, because roughly 95 % of pixels are background.

In [ ]:
LAYER_INDICES = {
    "terramind_v1_tiny": [2, 5, 8, 11], "terramind_v1_small": [2, 5, 8, 11],
    "terramind_v1_base": [2, 5, 8, 11], "terramind_v1_large": [5, 11, 17, 23],
}


def build_task(modalities, backbone=BACKBONE, lr=1e-4, freeze_backbone=True):
    model_args = dict(
        backbone=backbone,
        backbone_pretrained=True,
        backbone_modalities=list(modalities),
        backbone_bands={m: MODALITY_BANDS[m] for m in modalities},
        backbone_merge_method="mean",
        necks=[
            {"name": "SelectIndices", "indices": LAYER_INDICES[backbone]},
            {"name": "ReshapeTokensToImage", "remove_cls_token": False},
            {"name": "LearnedInterpolateToPyramidal"},
        ],
        decoder="UNetDecoder",
        decoder_channels=[512, 256, 128, 64],
        head_dropout=0.1,
        num_classes=NUM_CLASSES,
    )
    return SemanticSegmentationTask(
        model_args=model_args,
        model_factory="EncoderDecoderFactory",
        loss="dice",
        ignore_index=IGNORE_INDEX,
        lr=lr,
        optimizer="AdamW",
        optimizer_hparams={"weight_decay": 0.05},
        scheduler="ReduceLROnPlateau",
        scheduler_hparams={"factor": 0.5, "patience": 3},
        freeze_backbone=freeze_backbone,
        class_names=CLASS_NAMES,
        plot_on_val=False,
    )


def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


_probe = build_task(["S2L2A", "S1RTC"])
tot, tr = count_params(_probe.model)
enc_tot, enc_tr = count_params(_probe.model.encoder)
print(f"{BACKBONE}: {enc_tot/1e6:.1f} M encoder params ({enc_tr/1e6:.1f} M trainable)")
print(f"whole model: {tot/1e6:.1f} M params, {tr/1e6:.1f} M trainable (decoder + head)")

In [ ]:
# Dry run: one batch through the untrained model, to confirm shapes before spending GPU time.
_probe.eval()
with torch.no_grad():
    out = _probe.model({m: batch["image"][m] for m in ("S2L2A", "S1RTC")})
print("logits", tuple(out.output.shape), "  target", tuple(batch["mask"].shape))
assert out.output.shape[1] == NUM_CLASSES
assert out.output.shape[-2:] == batch["mask"].shape[-2:]
del _probe, out
print("shapes OK")

## 3. Training

Checkpointing, early stopping and the LR schedule all watch `val/loss`, so they cannot
disagree with one another. Logs and checkpoints go to `output/<run name>/`.

**Resumable.** Every epoch writes `checkpoints/last.ckpt` (weights, optimizer, scheduler and
early-stopping state). If the runtime dies, re-running the training cell picks each arm up
from that file; an arm that already finished (`DONE` marker) is skipped and its `best.ckpt`
reused. To retrain from scratch after changing `EPOCHS`, `BATCH_SIZE` or `NORMALISATION`,
delete `output/<run name>/` first.

In [ ]:
OUT_DIR = Path("output")


def run_experiment(name, modalities, epochs=EPOCHS, backbone=BACKBONE, lr=1e-4):
    L.seed_everything(SEED, workers=True)
    dm = SmallMinesDataModule(
        DATA_ROOT, SPLIT_CSV, modalities=modalities, mean=MEAN, std=STD,
        batch_size=BATCH_SIZE, num_workers=0 if SMOKE_TEST else 2,
        limit=16 if SMOKE_TEST else None,
    )
    task = build_task(modalities, backbone=backbone, lr=lr)

    run_dir = OUT_DIR / name
    best_path = run_dir / "checkpoints" / "best.ckpt"
    last_path = run_dir / "checkpoints" / "last.ckpt"
    done_marker = run_dir / "DONE"

    # Arm already finished on a previous attempt: skip training, reuse its best weights.
    if done_marker.exists() and best_path.exists():
        print(f"[{name}] already trained, reusing {best_path}")
        return dict(name=name, task=task, datamodule=dm, ckpt_path=str(best_path))

    # CSVLogger deletes an existing metrics.csv on start; keep the pre-crash epochs so
    # plot_curves can show the whole run.
    metrics_csv = run_dir / "metrics.csv"
    if metrics_csv.exists():
        n = len(list(run_dir.glob("metrics_part*.csv")))
        metrics_csv.rename(run_dir / f"metrics_part{n}.csv")

    logger = CSVLogger(OUT_DIR, name=name, version="")
    ckpt = ModelCheckpoint(monitor="val/loss", mode="min", save_top_k=1,
                           filename="best", save_last=True)
    trainer = L.Trainer(
        accelerator="auto", devices=1, max_epochs=epochs,
        precision="16-mixed" if DEVICE == "cuda" else "32-true",
        logger=logger, log_every_n_steps=10, enable_progress_bar=True,
        callbacks=[ckpt, LearningRateMonitor(logging_interval="epoch"),
                   EarlyStopping(monitor="val/loss", mode="min", patience=8)],
    )

    # Resume mid-run if a previous attempt was interrupted. last.ckpt carries the epoch
    # counter, optimizer, ReduceLROnPlateau and EarlyStopping state, so patience is not reset.
    resume = str(last_path) if last_path.exists() else None
    if resume:
        print(f"[{name}] resuming from {resume}")

    t0 = time.time()
    trainer.fit(task, datamodule=dm, ckpt_path=resume)
    done_marker.touch()
    print(f"[{name}] {trainer.current_epoch} epochs in {(time.time() - t0)/60:.1f} min, "
          f"best val loss {ckpt.best_model_score:.4f}")
    return dict(name=name, task=task, datamodule=dm, ckpt_path=ckpt.best_model_path)

In [ ]:
ARMS = {
    "A1_S2":    ["S2L2A"],
    "A4_S2_S1": ["S2L2A", "S1RTC"],
}

runs = {name: run_experiment(name, mods) for name, mods in ARMS.items()}

## 4. Evaluation

Reported on the **mining class only**. Two-class mIoU averages mining against a background
that is ~95 % of pixels, so it moves for reasons unrelated to the research question.

In [ ]:
def collect_predictions(task, loader):
    task.eval().to(DEVICE)
    preds, targets, names = [], [], []
    with torch.no_grad():
        for b in loader:
            x = {m: t.to(DEVICE) for m, t in b["image"].items()}
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                logits = task.model(x).output
            preds.append(logits.argmax(1).cpu())
            targets.append(b["mask"])
            names += list(b["filename"])
    return torch.cat(preds).numpy(), torch.cat(targets).numpy(), names


def mine_scores(pred, true):
    """Binary scores for the mining class over valid pixels."""
    valid = true != IGNORE_INDEX
    p = (pred[valid] == 1)
    t = (true[valid] == 1)
    tp, fp, fn = int((p & t).sum()), int((p & ~t).sum()), int((~p & t).sum())
    iou = tp / max(tp + fp + fn, 1)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    return dict(IoU=iou, F1=f1, precision=prec, recall=rec,
                mine_pixel_share=float(t.mean()), mine_pixels=int(t.sum()))


def evaluate(run):
    task, dm = run["task"], run["datamodule"]
    if run["ckpt_path"]:
        state = torch.load(run["ckpt_path"], map_location="cpu")["state_dict"]
        task.load_state_dict(state)
    pred, true, names = collect_predictions(task, dm.test_dataloader())
    return dict(scores=mine_scores(pred, true), pred=pred, true=true, names=names)


evals = {name: evaluate(run) for name, run in runs.items()}

table = pd.DataFrame({n: e["scores"] for n, e in evals.items()}).T
table.insert(0, "bands", [len(sum([MODALITY_BANDS[m] for m in ARMS[n]], [])) for n in table.index])
display(table.style.format({"IoU": "{:.3f}", "F1": "{:.3f}", "precision": "{:.3f}",
                            "recall": "{:.3f}", "mine_pixel_share": "{:.3%}",
                            "bands": "{:.0f}", "mine_pixels": "{:,.0f}"}))

d_iou = table.loc["A4_S2_S1", "IoU"] - table.loc["A1_S2", "IoU"]
d_f1 = table.loc["A4_S2_S1", "F1"] - table.loc["A1_S2", "F1"]
print(f"\nAdding Sentinel-1:  mine IoU {d_iou:+.4f}   mine F1 {d_f1:+.4f}")

OUT_DIR.mkdir(exist_ok=True)
table.to_csv(OUT_DIR / "arm_comparison.csv")
print("saved ->", (OUT_DIR / "arm_comparison.csv").resolve())

In [ ]:
def read_metrics(name):
    """metrics.csv plus any metrics_part*.csv left by interrupted attempts, in order."""
    parts = sorted((OUT_DIR / name).glob("metrics_part*.csv"),
                   key=lambda p: int(p.stem.split("part")[-1]))
    parts.append(OUT_DIR / name / "metrics.csv")
    return pd.concat([pd.read_csv(p) for p in parts if p.exists()], ignore_index=True)


def plot_curves(names):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    for name in names:
        log = read_metrics(name)
        ep = log.groupby("epoch").mean(numeric_only=True)
        if "train/loss" in ep: axes[0].plot(ep.index, ep["train/loss"], label=f"{name} train")
        if "val/loss" in ep:   axes[1].plot(ep.index, ep["val/loss"], label=name)
    axes[0].set_title("train dice loss"); axes[1].set_title("val dice loss")
    for ax in axes:
        ax.set_xlabel("epoch"); ax.grid(alpha=.3); ax.legend(fontsize=8)
    plt.tight_layout()


plot_curves(list(runs))

In [ ]:
# Predictions on the test samples with the most mining, both arms side by side.
meta = samples.set_index(samples.tile + "_" + samples.year)
order = meta.loc[evals["A4_S2_S1"]["names"], "class_percentage"].values.argsort()[::-1][:3]

for i in order:
    name = evals["A4_S2_S1"]["names"][i]
    tile, year = name.rsplit("_", 1)
    for arm in ARMS:
        show_sample(tile, year, pred=evals[arm]["pred"][i])
        plt.suptitle(f"{arm}  —  tile {tile} / {year}", fontsize=10)

## 5. Optional — is there any cloud range in this dataset?

Ten minutes, and it decides whether a cloud-stratified analysis is possible at all. The proxy
is mean blue reflectance over valid **non-mine** pixels: computed off-mine because bright
alluvial mining ponds mimic cloud in the visible bands, so a whole-tile brightness measure
would partly be measuring the target class.

If the histogram is narrow, the honest write-up is "the available range was too small to
resolve a trend" — and knowing that now costs an hour rather than a chapter.

In [ ]:
import rasterio
from smallmines_data import patch_paths

test_samples = datamodule.samples["test"]
vals = []
for tile, year in test_samples:
    ip, mp = patch_paths(DATA_ROOT, year, tile)
    with rasterio.open(ip) as s: img = s.read().astype(np.float32)
    with rasterio.open(mp) as s: msk = s.read(1)
    vals.append(haze_proxy(img, msk))

vals = np.array(vals, dtype=float)
vals = vals[np.isfinite(vals)]
plt.figure(figsize=(5.5, 3.2))
plt.hist(vals, bins=40, color="#0D6E73")
plt.xlabel("mean blue reflectance over non-mine pixels")
plt.ylabel("test samples"); plt.grid(alpha=.3)
plt.title("haze proxy distribution")
plt.tight_layout()
print(f"n={len(vals)}  min {vals.min():.0f}  p25 {np.percentile(vals,25):.0f}  "
      f"median {np.median(vals):.0f}  p75 {np.percentile(vals,75):.0f}  max {vals.max():.0f}")
print(f"interquartile range / median = {(np.percentile(vals,75)-np.percentile(vals,25))/np.median(vals):.2f}")

## 6. Notes for the write-up

- **What was tested.** TerraMind's *pretrained representation* — the encoder was never
  updated. Every number is "how separable is mining in frozen TerraMind feature space",
  not "how good can a TerraMind-based model get". A fine-tune (`freeze_backbone=False`,
  `lr=2e-5`) is the natural follow-up.
- **Why a new split.** The published per-year splits were stratified independently and
  disagree for 42 % of tiles. Pooling both years on them would put the same ground in train
  and test. The split here is by tile, seeded, stratified on mean mine coverage — so these
  numbers are *not* directly comparable to the per-year benchmarks in the dataset paper.
  One single-year run on the published split would restore that comparison.
- **Radar offset — read before interpreting a null.** With TerraMind's S1RTC statistics this
  dataset's VV/VH sit roughly 2 σ above zero: its backscatter is brighter than TerraMind's
  pretraining average. If A4 does not beat A1, re-run with `NORMALISATION = "dataset"` before
  concluding anything. "SAR does not help" and "SAR was presented out of distribution to a
  frozen encoder" are different claims and only one of them is about radar.
- **DEM is close to dead here.** Standardised against TerraMind's global elevation statistics
  (mean ≈ 670 m, std ≈ 950 m), south-western Ghana's low relief compresses to a near-constant
  channel — the band diagnostic in §1.3 prints its standard deviation. That is why DEM is not
  in either arm: a DEM effect measured this way would be an artefact of normalisation.
- **Metrics.** Mining-class IoU and F1, with the mining pixel share reported alongside so the
  reader can see how thin the positive class is. Two-class mIoU is not used.
- **Merge method.** `backbone_merge_method="mean"` averages per-modality tokens, so a
  weakly-encoded radar channel can dilute rather than add. If A4 is flat, trying `"concat"`
  is the second thing to check after normalisation.
- **Resolution.** 128 × 128 at 10 m with 16-px patches gives an 8 × 8 token grid; the decoder
  recovers 16× spatial detail. Upsampling inputs to 224 × 224 is an easy variant if pit
  boundaries look coarse.